# Overfitting Investigation: PPO Log Return Model

This notebook provides detailed overfitting analysis for the PPO model with log return reward.

**Model Analyzed:**
- PPO with `log_return` reward: `log(V_t+1) - log(V_t)`
- RecurrentPPO with LSTM (hidden_size=50)
- Walk-forward validation with 3 folds
- 26 features (19 ticker + 5 macro + 2 calendar)

**Analysis Components:**

**Part 1:** Learning Curves Analysis
- Train vs validation Sharpe per fold
- Peak detection and early stopping analysis
- Train-validation performance gaps

**Part 2:** Overfitting Assessment
- Peak position analysis (early peaking = overfitting)
- Decline from peak analysis
- Cross-fold consistency

**Part 3:** Validation-Test Gap Analysis
- Compare validation Sharpe to test Sharpe
- Generalization assessment

**Part 4:** Recommendations
- Training improvements
- Early stopping optimization
- Production deployment guidance

## Setup & Imports

In [1]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

# Setup - notebooks are in project root
project_root = Path.cwd()
sys.path.insert(0, str(project_root))

plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Import config
from config import N_FOLDS, FEATURE_COLS, OUTPUT_DIR

# Import viz functions
from viz import (
    # Individual model analysis
    plot_learning_curves,
    analyze_peak_positions,
    plot_train_val_comparison,
    plot_validation_test_gap,
    # Multi-model comparison (for fold comparison)
    plot_multi_model_learning_curves,
    compare_overfitting_metrics,
    plot_multi_model_val_test_gap,
    create_overfitting_summary_table
)

# Paths
DATA_DIR = project_root / 'data' / 'processed'
MODELS_DIR = OUTPUT_DIR  # Use unified models directory from config
RESULTS_DIR = project_root / 'results' / 'backtests'
ANALYSIS_OUTPUT_DIR = project_root / 'results' / 'overfitting_analysis'
ANALYSIS_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Model configuration - log_return reward
PPO_REWARD_TYPES = ['log_return']

print(f"✓ Setup complete")
print(f"✓ Timestamp: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
print(f"✓ Project root: {project_root}")
print(f"✓ Models dir: {MODELS_DIR}")
print(f"✓ Output directory: {ANALYSIS_OUTPUT_DIR}")
print(f"✓ Features: {len(FEATURE_COLS)} features")
print(f"✓ Folds: {N_FOLDS}")

✓ Loaded 26 features from metadata_weekly.json
  Using MINIMAL feature set (evidence-based)
✓ Setup complete
✓ Timestamp: 2025-11-20 18:41:11
✓ Project root: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio
✓ Models dir: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models
✓ Output directory: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/results/overfitting_analysis
✓ Features: 26 features
✓ Folds: 3


## Load Training Logs for All Models

In [2]:
def load_model_logs(reward_type: str):
    """
    Load training and validation logs for the PPO model.
    
    Parameters:
    -----------
    reward_type : str
        Reward type ('log_return')
        
    Returns:
    --------
    tuple : (train_logs, val_logs) dictionaries mapping fold_idx to DataFrames
    """
    # Use unified models directory (models/fold_0, models/fold_1, etc.)
    model_dir = MODELS_DIR
    
    train_logs = {}
    val_logs = {}
    
    for fold_idx in range(N_FOLDS):
        fold_dir = model_dir / f'fold_{fold_idx}'
        
        # Check for logs in fold directory
        train_path = fold_dir / 'logs' / 'train_episodes.csv'
        val_path = fold_dir / 'logs' / 'val_evaluations.csv'
        
        # Fallback to direct fold directory if logs subdir doesn't exist
        if not train_path.exists():
            train_path = fold_dir / 'train_episodes.csv'
        if not val_path.exists():
            val_path = fold_dir / 'val_evaluations.csv'
        
        if train_path.exists():
            train_logs[fold_idx] = pd.read_csv(train_path)
        else:
            print(f"  ⚠️  Train log not found: {train_path}")
        
        if val_path.exists():
            val_logs[fold_idx] = pd.read_csv(val_path)
        else:
            print(f"  ⚠️  Val log not found: {val_path}")
    
    return train_logs, val_logs

# Load logs for log_return reward model
print("Loading training logs for PPO log_return reward model...\n")

all_models_data = {}

for reward_type in PPO_REWARD_TYPES:
    print(f"Loading {reward_type}...")
    train_logs, val_logs = load_model_logs(reward_type)
    
    all_models_data[reward_type] = {
        'train': train_logs,
        'val': val_logs
    }
    
    print(f"  ✓ Train folds: {len(train_logs)}")
    print(f"  ✓ Val folds: {len(val_logs)}")

print(f"\n✓ Loaded logs for {len(PPO_REWARD_TYPES)} reward model(s)")

Loading training logs for PPO log_return reward model...

Loading log_return...
  ⚠️  Train log not found: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models/fold_0/train_episodes.csv
  ⚠️  Val log not found: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models/fold_0/val_evaluations.csv
  ⚠️  Train log not found: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models/fold_1/train_episodes.csv
  ⚠️  Val log not found: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models/fold_1/val_evaluations.csv
  ⚠️  Train log not found: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_finance/rl_relabalance_portfolio/models/fold_2/train_episodes.csv
  ⚠️  Val log not found: /Users/nataliamarko/Documents/GitHub_Projects/reinforcement_learning_in_

---
## Part 1: Learning Curves Analysis

Analyze PPO unified reward model using the well-structured viz functions.

### 1.1 PPO Unified Reward - Learning Curves

In [3]:
reward_type = 'log_return'
model_data = all_models_data[reward_type]

print(f"{'='*80}")
print(f"PPO - {reward_type.upper()} REWARD")
print(f"{'='*80}\n")

# Create model-specific output directory
model_output_dir = ANALYSIS_OUTPUT_DIR / f"ppo_{reward_type}"
model_output_dir.mkdir(parents=True, exist_ok=True)

# Plot learning curves
print("\n1. Learning Curves Analysis:")
plot_learning_curves(model_data['train'], model_data['val'], model_output_dir)

# Analyze peak positions
print("\n2. Peak Position Analysis:")
peak_df_log_return = analyze_peak_positions(model_data['val'], model_output_dir)

# Train vs validation comparison
print("\n3. Train vs Validation Comparison:")
plot_train_val_comparison(model_data['train'], model_data['val'], model_output_dir)

print(f"\n✓ Learning curves analysis complete for {reward_type}")
print(f"✓ Results saved to: {model_output_dir}")

PPO - LOG_RETURN REWARD


1. Learning Curves Analysis:


ValueError: Number of rows must be a positive integer, not 0

<Figure size 1400x0 with 0 Axes>

---
## Part 2: Overfitting Assessment

In [ ]:
# Detailed overfitting assessment per fold

print(f"\n{'='*80}")
print("OVERFITTING ASSESSMENT - LOG_RETURN REWARD")
print(f"{'='*80}\n")

models_peak_analysis = {
    'log_return': peak_df_log_return
}

for reward_type, peak_df in models_peak_analysis.items():
    print(f"\n{reward_type.upper()} REWARD - FOLD-BY-FOLD ANALYSIS:")
    print("-" * 80)
    
    for _, row in peak_df.iterrows():
        fold = int(row['fold'])
        peak_pos = row['peak_position']
        decline_pct = row['decline_pct']
        
        signals = []
        if peak_pos < 0.5:
            signals.append('Early Peak')
        if decline_pct > 20:
            signals.append('Large Decline')
        
        risk = 'HIGH' if len(signals) >= 2 else 'MEDIUM' if len(signals) == 1 else 'LOW'
        
        risk_emoji = {'LOW': '🟢', 'MEDIUM': '🟡', 'HIGH': '🔴'}[risk]
        
        print(f"\n  Fold {fold}: {risk_emoji} {risk} risk")
        print(f"    Peak at {peak_pos*100:.1f}% of training")
        print(f"    Decline from peak: {decline_pct:.1f}%")
        if signals:
            print(f"    Risk signals: {', '.join(signals)}")
        else:
            print(f"    Risk signals: None")

# Summary statistics
avg_peak = peak_df_log_return['peak_position'].mean()
avg_decline = peak_df_log_return['decline_pct'].mean()

print(f"\n{'='*80}")
print("SUMMARY STATISTICS")
print(f"{'='*80}")
print(f"  Average peak position: {avg_peak*100:.1f}%")
print(f"  Average decline from peak: {avg_decline:.1f}%")

if avg_peak > 0.6:
    print(f"  ✅ Peak timing is good (late in training)")
elif avg_peak > 0.4:
    print(f"  🟡 Peak timing is moderate")
else:
    print(f"  🔴 Early peaking detected (potential overfitting)")

if avg_decline < 15:
    print(f"  ✅ Validation performance is stable")
elif avg_decline < 30:
    print(f"  🟡 Some instability in validation performance")
else:
    print(f"  🔴 High instability (possible overfitting)")

print(f"{'='*80}")


OVERFITTING ASSESSMENT - LOG_RETURN REWARD


LOG_RETURN REWARD - FOLD-BY-FOLD ANALYSIS:
--------------------------------------------------------------------------------

  Fold 0: 🔴 HIGH risk
    Peak at 14.3% of training
    Decline from peak: 52.7%
    Risk signals: Early Peak, Large Decline

  Fold 1: 🔴 HIGH risk
    Peak at 0.0% of training
    Decline from peak: 24.2%
    Risk signals: Early Peak, Large Decline

  Fold 2: 🔴 HIGH risk
    Peak at 25.0% of training
    Decline from peak: 79.3%
    Risk signals: Early Peak, Large Decline

SUMMARY STATISTICS
  Average peak position: 13.1%
  Average decline from peak: 52.1%
  🔴 Early peaking detected (potential overfitting)
  🔴 High instability (possible overfitting)


---
## Part 3: Validation-Test Gap Analysis

Compare validation Sharpe to test Sharpe to assess generalization.

In [ ]:
# Load backtest results
backtest_summary_path = RESULTS_DIR / 'backtest_summary_all_models.csv'

if backtest_summary_path.exists():
    backtest_df = pd.read_csv(backtest_summary_path)
    print(f"✓ Loaded backtest results from: {backtest_summary_path}\n")
    
    # Extract test Sharpe for PPO models
    models_test_results = {}
    
    for reward_type in PPO_REWARD_TYPES:
        # Find test results for PPO models from this reward type
        # Models are named like "PPO_fold0", "PPO_fold1", etc.
        ppo_rows = backtest_df[backtest_df['Model'].str.startswith('PPO_fold')]
        
        if len(ppo_rows) > 0:
            # Get fold results from training
            fold_results = []
            for fold_idx in range(N_FOLDS):
                fold_dir = MODELS_DIR / f'fold_{fold_idx}'
                val_path = fold_dir / 'logs' / 'val_evaluations.csv'
                
                # Fallback path
                if not val_path.exists():
                    val_path = fold_dir / 'val_evaluations.csv'
                
                if val_path.exists():
                    val_df = pd.read_csv(val_path)
                    best_val = val_df['sharpe_ratio'].max()
                    final_val = val_df['sharpe_ratio'].iloc[-1]
                    
                    # Get corresponding test Sharpe
                    model_name = f"PPO_fold{fold_idx}"
                    test_row = backtest_df[backtest_df['Model'] == model_name]
                    test_sharpe = test_row['Test Sharpe'].values[0] if len(test_row) > 0 else np.nan
                    
                    fold_results.append({
                        'fold': fold_idx,
                        'best_val_sharpe': best_val,
                        'final_val_sharpe': final_val,
                        'test_sharpe': test_sharpe
                    })
            
            if fold_results:
                fold_results_df = pd.DataFrame(fold_results)
                
                # Find best fold by best validation Sharpe
                best_fold_idx = fold_results_df['best_val_sharpe'].idxmax()
                tested_fold = fold_results_df.loc[best_fold_idx, 'fold']
                best_test_sharpe = fold_results_df.loc[best_fold_idx, 'test_sharpe']
                
                models_test_results[reward_type] = {
                    'fold_results': fold_results_df,
                    'test_sharpe': best_test_sharpe,
                    'tested_fold': tested_fold
                }
                
                print(f"{reward_type.upper()}: Test Sharpe = {best_test_sharpe:.3f} (Best Fold {tested_fold})")
                print(f"  All folds: {', '.join([f'Fold {int(r[\"fold\"])}: {r[\"test_sharpe\"]:.3f}' for _, r in fold_results_df.iterrows()])}")
    
    has_test_results = True
else:
    print(f"⚠️  Backtest results not found: {backtest_summary_path}")
    print("   Run the backtest notebook (03_backtests_all_models.ipynb) first.")
    has_test_results = False

### 3.1 Validation-Test Gap Analysis

In [ ]:
if has_test_results and models_test_results:
    print("\n" + "="*100)
    print("VALIDATION-TEST GAP ANALYSIS")
    print("="*100)
    
    for reward_type, data in models_test_results.items():
        fold_results = data['fold_results']
        test_sharpe = data['test_sharpe']
        tested_fold = data['tested_fold']
        
        print(f"\n{'-'*100}")
        print(f"{reward_type.upper()} REWARD")
        print(f"{'-'*100}")
        
        # Show all folds
        print(f"\n1. VALIDATION PERFORMANCE BY FOLD:")
        for _, row in fold_results.iterrows():
            fold = int(row['fold'])
            best = row['best_val_sharpe']
            final = row['sharpe_ratio']
            is_tested = " (TESTED)" if fold == tested_fold else ""
            print(f"   Fold {fold}: Best={best:.3f}, Final={final:.3f}{is_tested}")
        
        # Best fold analysis
        tested_row = fold_results[fold_results['fold'] == tested_fold].iloc[0]
        best_sharpe = tested_row['best_val_sharpe']
        final_sharpe = tested_row['sharpe_ratio']
        
        print(f"\n2. BEST FOLD ({tested_fold}) PERFORMANCE:")
        print(f"   Best Sharpe (checkpoint): {best_sharpe:.3f}")
        print(f"   Final Sharpe (end): {final_sharpe:.3f}")
        
        decline = best_sharpe - final_sharpe
        decline_pct = (decline / best_sharpe * 100) if best_sharpe != 0 else 0
        print(f"   Decline during training: {decline:.3f} ({decline_pct:.1f}%)")
        
        print(f"\n3. TEST PERFORMANCE:")
        print(f"   Test Sharpe: {test_sharpe:.3f}")
        
        print(f"\n4. OVERFITTING GAPS:")
        gap_from_best = ((test_sharpe - best_sharpe) / best_sharpe * 100) if best_sharpe != 0 else 0
        gap_from_final = ((test_sharpe - final_sharpe) / final_sharpe * 100) if final_sharpe != 0 else 0
        
        print(f"   From Best Val to Test:  {best_sharpe:.3f} → {test_sharpe:.3f} = {gap_from_best:+.1f}%")
        print(f"   From Final Val to Test: {final_sharpe:.3f} → {test_sharpe:.3f} = {gap_from_final:+.1f}%")
        
        print(f"\n5. INTERPRETATION:")
        abs_gap = abs(gap_from_best)
        if abs_gap > 50:
            print(f"   🔴 SEVERE OVERFITTING ({gap_from_best:.0f}% gap)")
            print(f"      Model learned validation-specific patterns.")
        elif abs_gap > 30:
            print(f"   🟠 MODERATE OVERFITTING ({gap_from_best:.0f}% gap)")
            print(f"      Significant generalization gap.")
        elif abs_gap > 15:
            print(f"   🟡 MILD OVERFITTING ({gap_from_best:.0f}% gap)")
            print(f"      Some generalization issues.")
        elif gap_from_best > 0:
            print(f"   🟢 EXCELLENT GENERALIZATION (+{gap_from_best:.0f}% improvement)")
            print(f"      Model performs better on test than validation!")
        else:
            print(f"   🟢 GOOD GENERALIZATION ({gap_from_best:.0f}% gap)")
            print(f"      Model generalizes well to unseen data.")
    
    print(f"\n{'='*100}")
else:
    print("\n⚠️  No test results available for gap analysis")


⚠️  No test results available for gap analysis


---
## Part 4: Recommendations & Conclusions

### 4.1 Visual Summary

In [ ]:
# Visual summary of overfitting metrics

if has_test_results and models_test_results:
    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    
    # Get log_return model data
    log_return_data = models_test_results['log_return']
    fold_results = log_return_data['fold_results']
    test_sharpe = log_return_data['test_sharpe']
    
    # Plot 1: Validation Sharpe per fold
    ax1 = axes[0]
    folds = fold_results['fold'].values
    best_vals = fold_results['best_val_sharpe'].values
    final_vals = fold_results['final_val_sharpe'].values
    test_sharpes = fold_results['test_sharpe'].values
    
    x = np.arange(len(folds))
    width = 0.25
    
    bars1 = ax1.bar(x - width, best_vals, width, label='Best Val', color='#2ecc71', alpha=0.8)
    bars2 = ax1.bar(x, final_vals, width, label='Final Val', color='#3498db', alpha=0.8)
    bars3 = ax1.bar(x + width, test_sharpes, width, label='Test', color='#e74c3c', alpha=0.8)
    
    ax1.set_xlabel('Fold', fontweight='bold')
    ax1.set_ylabel('Sharpe Ratio', fontweight='bold')
    ax1.set_title('Validation vs Test Sharpe', fontweight='bold')
    ax1.set_xticks(x)
    ax1.set_xticklabels([f'Fold {int(f)}' for f in folds])
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # Plot 2: Peak position per fold
    ax2 = axes[1]
    peak_positions = peak_df_log_return['peak_position'].values * 100
    bars = ax2.bar(x, peak_positions, color='#9b59b6', alpha=0.8)
    ax2.axhline(y=50, color='red', linestyle='--', linewidth=2, label='Midpoint')
    
    ax2.set_xlabel('Fold', fontweight='bold')
    ax2.set_ylabel('Peak Position (%)', fontweight='bold')
    ax2.set_title('Peak Timing Analysis', fontweight='bold')
    ax2.set_xticks(x)
    ax2.set_xticklabels([f'Fold {int(f)}' for f in folds])
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    ax2.set_ylim([0, 100])
    
    # Plot 3: Decline from peak per fold
    ax3 = axes[2]
    declines = peak_df_log_return['decline_pct'].values
    colors = ['#2ecc71' if d < 15 else '#f39c12' if d < 30 else '#e74c3c' for d in declines]
    bars = ax3.bar(x, declines, color=colors, alpha=0.8)
    ax3.axhline(y=20, color='red', linestyle='--', linewidth=2, label='Warning (20%)')
    
    ax3.set_xlabel('Fold', fontweight='bold')
    ax3.set_ylabel('Decline (%)', fontweight='bold')
    ax3.set_title('Decline from Peak', fontweight='bold')
    ax3.set_xticks(x)
    ax3.set_xticklabels([f'Fold {int(f)}' for f in folds])
    ax3.legend()
    ax3.grid(True, alpha=0.3)
    
    plt.tight_layout()
    summary_path = ANALYSIS_OUTPUT_DIR / 'log_return_overfitting_summary.png'
    plt.savefig(summary_path, dpi=300, bbox_inches='tight')
    plt.show()
    
    print(f"\n💾 Summary plot saved: {summary_path}")
else:
    print("\n⚠️  No results available for visualization")

### 4.2 Final Recommendations

In [ ]:
print("\n" + "="*100)
print("RECOMMENDATIONS & CONCLUSIONS")
print("="*100)

# Calculate overfitting score
avg_peak_pos = peak_df_log_return['peak_position'].mean()
avg_decline = peak_df_log_return['decline_pct'].mean()

# Early peak contributes 50%, large decline contributes 50%
peak_score = max(0, (0.5 - avg_peak_pos)) * 200  # 0-100 scale
decline_score = min(100, avg_decline)  # 0-100 scale
overfitting_score = (peak_score + decline_score) / 2

if overfitting_score < 30:
    status = "🟢 EXCELLENT"
elif overfitting_score < 50:
    status = "🟡 GOOD"
elif overfitting_score < 70:
    status = "🟠 FAIR"
else:
    status = "🔴 POOR"

print(f"\n🏆 LOG_RETURN REWARD MODEL ASSESSMENT:")
print("-" * 100)
print(f"   Generalization Score: {status} (score: {overfitting_score:.1f}/100)")
print(f"   Average peak position: {avg_peak_pos*100:.1f}%")
print(f"   Average decline from peak: {avg_decline:.1f}%")

# Initialize val_test_gap
val_test_gap = 0.0

if has_test_results and models_test_results:
    log_return_data = models_test_results['log_return']
    fold_results_df = log_return_data['fold_results']
    
    # Calculate average metrics across all folds
    avg_best_val = fold_results_df['best_val_sharpe'].mean()
    avg_test = fold_results_df['test_sharpe'].mean()
    
    val_test_gap = ((avg_test - avg_best_val) / avg_best_val * 100) if avg_best_val != 0 else 0
    
    print(f"   Avg best validation Sharpe: {avg_best_val:.3f}")
    print(f"   Avg test Sharpe: {avg_test:.3f}")
    print(f"   Val-Test gap: {val_test_gap:+.1f}%")

print(f"\n📋 RECOMMENDATIONS:")
print("-" * 100)

# Recommendations based on analysis
if avg_peak_pos < 0.4:
    print(f"   1. ⚠️  Early Stopping: Model peaks early - reduce training duration")
    print(f"      Current peak at {avg_peak_pos*100:.0f}% - consider stopping at 50-60%")
else:
    print(f"   1. ✅ Training Duration: Peak timing is good ({avg_peak_pos*100:.0f}%)")

if avg_decline > 30:
    print(f"   2. ⚠️  Stability: High decline ({avg_decline:.0f}%) - increase regularization")
    print(f"      Consider: higher entropy coefficient, dropout, or L2 penalty")
elif avg_decline > 15:
    print(f"   2. 🟡 Stability: Moderate decline ({avg_decline:.0f}%) - monitor closely")
else:
    print(f"   2. ✅ Stability: Validation performance is stable ({avg_decline:.0f}%)")

if has_test_results and abs(val_test_gap) > 30:
    print(f"   3. ⚠️  Generalization: Large val-test gap ({val_test_gap:+.0f}%)")
    print(f"      Consider: more diverse training data, stronger regularization")
elif has_test_results:
    print(f"   3. ✅ Generalization: Good val-test consistency ({val_test_gap:+.0f}%)")
else:
    print(f"   3. ⚠️  No test results available - run backtests first")

print(f"\n📊 SUMMARY:")
print("-" * 100)
print(f"   Model: PPO with log_return reward")
print(f"   Architecture: RecurrentPPO (LSTM hidden_size=50)")
print(f"   Features: {len(FEATURE_COLS)} features (26: 19 ticker + 5 macro + 2 calendar)")
print(f"   Folds analyzed: {N_FOLDS}")
print(f"   Output directory: {ANALYSIS_OUTPUT_DIR}")

print(f"\n💡 KEY TAKEAWAYS:")
print("-" * 100)
print(f"   • Log return reward: log(V_t+1) - log(V_t)")
print(f"   • Walk-forward cross-validation with {N_FOLDS} folds")
print(f"   • Early stopping based on validation Sharpe prevents overfitting")
print(f"   • RecurrentPPO with LSTM captures temporal dependencies")

print(f"\n{'='*100}")
print("✅ OVERFITTING INVESTIGATION COMPLETE")
print(f"{'='*100}")

## Notes

**Analysis Performed:**
- Learning curves analysis (train vs validation Sharpe per fold)
- Peak position analysis (early peaking = overfitting)
- Decline from peak analysis (validation instability)
- Validation-test gap analysis (generalization assessment)
- Cross-fold consistency evaluation

**Model Configuration:**
- Reward: Log return (`log(V_t+1) - log(V_t)`)
- Architecture: RecurrentPPO with LSTM (hidden_size=50)
- Features: 26 (19 ticker + 5 macro + 2 calendar) loaded from metadata_weekly.json
- Walk-forward validation with 3 folds
- Data: Weekly rebalancing (2015-2025)

**Visualization Functions Used:**
- `plot_learning_curves` - Train vs validation over time
- `analyze_peak_positions` - Peak detection and timing
- `plot_train_val_comparison` - Performance gaps analysis

**Output Files:**
- Learning curve plots in `results/overfitting_analysis/ppo_log_return/`
- Summary plot: `log_return_overfitting_summary.png`

**Interpretation Guide:**
- **Peak Position < 50%**: Model overfits early in training
- **Decline > 20%**: Unstable validation performance
- **Val-Test Gap > 30%**: Significant generalization issues
- **Lower overfitting score**: Better generalization (0-100 scale)

**Key Configuration:**
- 26 features: momentum, volatility, technical, volume, risk, macro, calendar
- Log return reward encourages consistent compounding growth
- Early stopping based on validation Sharpe (patience=3)
- Walk-forward cross-validation prevents look-ahead bias

**Next Steps:**
1. If overfitting detected: Adjust hyperparameters (increase regularization)
2. If early peaking: Reduce training duration or adjust learning rate
3. If unstable: Increase entropy coefficient or add dropout
4. Monitor test performance for production deployment readiness